# numbers.tex

Every number the poster and the paper quote in their text, as LaTeX macros (`m`).

In [ ]:
from pathlib import Path

import numpy as np

import utils as u

data_path = Path("../data")
figs_path = Path("figs")

In [ ]:
u.load_data(data_path)
figs_path.mkdir(exist_ok=True)

In [ ]:
PLUS_VLAS = {"Pi": "vla:pi05_libero", "Xvla": "vla:xvla@lerobot/xvla-libero", "Smol": "vla:smolvla_libero",
             "Fast": "vla:pi0_fast@lerobot/pi0fast-libero", "PiZero": "vla:pi0_libero",
             "Jepa": "vla:vla_jepa@lerobot/VLA-JEPA-LIBERO"}  # fmt: skip

## Success on LIBERO and LIBERO-plus

In [ ]:
plus, control, abf = u.DATA["libero_plus"], u.DATA["libero_plus_control"], u.DATA["ablations_full"]
knn_plus = plus["knn:outputs/knn/v2_39.json"]
m = {
    "KnnAvg": u.f1(u.libero_success("knn:00_base")[0]),
    "KnnPlainResnet": u.f1(u.libero_success("knn:plain_resnet18_l3")[0]),
    "KnnLearnt": u.f1(u.libero_success("knn:plain_resnet18_l3_learned")[0]),
    "KnnFullLearnt": u.f1(u.libero_success("knn:full_resnet18_l3_learned")[0]),
    "KnnPiZeroBaseLt": u.f1(u.libero_success("knn:plain_pi0_base_grid2@L12_z")[0]),
    "KnnPiZeroLt": u.f1(u.libero_success("knn:plain_pi0_libero_grid2@L12_z")[0]),
    "AblNoLang": u.f1(abf["no_language"]["success"]),
    "AblPixels": u.f1(abf["encoder_pixels"]["success"]),
    "DemosTen": u.f1(abf["demos_10"]["success"]),
    "PlusKnn": u.f1(100 * knn_plus["all"]),
    "PlusN": str(knn_plus["n_episodes"]),
    "PlusPerCat": str(knn_plus["n_episodes"] // len(u.PLUS_CATEGORIES)),
    "PlusXvlaCamera": f"{100 * plus[PLUS_VLAS['Xvla']]['category/Camera Viewpoints']:.0f}",
}
for tag in ("Smol", "Fast", "PiZero"):  # reworded instructions vs the original wording, same scenes
    m[f"Plus{tag}Language"] = f"{100 * plus[PLUS_VLAS[tag]]['category/Language Instructions']:.0f}"
    m[f"Plus{tag}Ctrl"] = f"{100 * control[PLUS_VLAS[tag]]:.0f}"

## Drop under perturbation, and the kNN's rank among the VLAs

In [ ]:
drops = {lab: u.libero_success(run)[0] - u.plus_success(spec)[0] for lab, _role, run, spec in u.POLICIES}
vla_drops = sorted(drops[lab] for lab, role, *_ in u.POLICIES if role == "vla")
m.update(KnnDrop=u.f1(drops["kNN, full method"]), MinVlaDrop=u.f1(vla_drops[0]), MaxVlaDrop=u.f1(vla_drops[-1]),
         VlaN=str(len(vla_drops)))  # fmt: skip
entrants = [p for p in u.POLICIES if p[1] == "vla" or p[0] == "kNN, full method"]
for macro, score in (
    ("KnnRankLibero", lambda p: u.libero_success(p[2])[0]),
    ("KnnRankPlus", lambda p: u.plus_success(p[3])[0]),
):
    names = [p[0] for p in sorted(entrants, key=score, reverse=True)]
    m[macro] = str(1 + names.index("kNN, full method"))
m["RankN"] = str(len(entrants))
best_plus = 0  # LIBERO-plus suites where a kNN row is the best policy
for j in range(1, 5):
    vals = [(u.plus_success(spec)[j], role) for _lab, role, _run, spec in u.POLICIES]
    top = max(v for v, _ in vals)
    best_plus += any(v >= top - 1e-9 and role != "vla" for v, role in vals)
m["KnnBestPlusSuites"] = ["no", "one", "two", "three", "all four"][best_plus]

## Do the policies fail on the same episodes and tasks?

In [ ]:
fo = u.failure_overlap("libero")
lifts = [c / base for _lab, c, base in fo]
tc = sorted(u.task_corr("libero"))
m.update(FailBase=f"{100 * fo[0][2]:.0f}", FailMin=f"{100 * min(c for _l, c, _b in fo):.0f}",
         FailMax=f"{100 * max(c for _l, c, _b in fo):.0f}", LiftMin=f"{min(lifts):.1f}", LiftMax=f"{max(lifts):.1f}",
         TaskCorrMedian=f"{tc[len(tc) // 2]:.2f}")  # fmt: skip

## Backbone and action head

The lookup on each VLA's VLM before / after fine-tuning vs the VLA.

In [ ]:
rows = u.backbone_rows()
tuned = [r for r in rows if r[1] != r[2]]  # GR00T froze its VLM: before = after
up = [after[1] - before[1] for _l, before, after, _v in tuned if after[1] > before[1]]
subtract = [after[1] - vla[1] for _l, _b, after, vla in rows if after[1] >= vla[1]]
m.update(KeyN=str(len(rows)), BeatenClean=str(sum(after[0] >= vla[0] for _l, _b, after, vla in rows)),
         KeyNPlus=str(len(rows)), BeatenPlus=str(len(subtract)), FtN=str(len(tuned)), FtUp=str(len(up)),
         FtUpMin=f"{min(up):.0f}", FtUpMax=f"{max(up):.0f}",
         FtCleanUp=str(sum(after[0] > before[0] + 1 for _l, before, after, _v in tuned)),
         SubtractMax=f"{max(subtract):.0f}")  # fmt: skip
lookups = {lab: (r2, demo) for lab, r2, demo, _ratio in u.lookup_rows()}
m["LookupPi"] = f"{lookups['π0.5'][0]:.2f}"
m["LookupHuman"] = f"{np.mean([demo for _r2, demo in lookups.values()]):.2f}"
for tag in ("Pi", "Jepa", "Smol", "Fast"):  # the best lookup in the VLA's own key spaces on LIBERO-plus rollouts
    r2 = u.DATA["head_vs_knn_plus"][PLUS_VLAS[tag][4:]]["r2_k8"]
    m[f"HeadPlus{tag}"] = f"{max(v for k, v in r2.items() if k != 'resnet18_l3'):.2f}"

## Local action consistency

In [ ]:
nm = u.DATA["neighbor_metric"]
fair = nm["correlation"]["fair"]
shown = [*u.NEIGHBOR_KEYS, *(k["label"] for k in u.DATA["extra_keys"])]  # one key per backbone, as in neighbors.ipynb
others = [nm["spaces"][k]["lac@8"] for k in shown if k != "π0 LIBERO L12"]
m.update(NmOtherMin=f"{min(others):.2f}", NmOtherMax=f"{max(others):.2f}")
m.update(NmFairRho=f"{fair['spearman_lac@8']:.2f}", NmFairR=f"{fair['pearson_lac@8']:.2f}", NmFairN=str(fair["n"]),
         NmPiZeroLibero=f"{nm['spaces']['π0 LIBERO L12']['lac@8']:.2f}")  # fmt: skip

## Embedding maps

Same-task neighbours on LIBERO-Goal (one scene, ten instructions) and the phase gap.

In [ ]:
spaces = u.MAPS["spaces"]
goal = {name: 100 * v["task_nn/libero_goal"] for name, v in spaces.items()}
sees_prompt = [n for n in goal if n.startswith(("π0", "X-VLA"))]  # bidirectional prefix attention over image + text
causal = [n for n in goal if spaces[n]["stage"] in ("before", "after") and n not in sees_prompt]
pairs = [(b, a) for b in spaces for a in spaces if spaces[b]["family"] == spaces[a]["family"]
         and spaces[b]["stage"] == "before" and spaces[a]["stage"] == "after"]  # fmt: skip
phase = {n: v["progress_gap"] for n, v in spaces.items()}
m.update(MapN=f"{len(u.MAPS['progress']):,}", MapGoalResnet=f"{goal['ResNet-18 grid']:.0f}",
         MapGoalPromptMin=f"{min(goal[n] for n in sees_prompt):.0f}", MapGoalCausalMin=f"{min(goal[n] for n in causal):.0f}",
         MapGoalCausalMax=f"{max(goal[n] for n in causal):.0f}", MapPhaseN=str(len(pairs)),
         MapPhaseDown=str(sum(phase[a] < phase[b] for b, a in pairs)),
         MapPhaseEvoBefore=f"{phase['Evo-1 base grid2@L10']:.3f}", MapPhaseEvoAfter=f"{phase['Evo-1 LIBERO grid2@L10']:.3f}",
         MapPhasePiZeroBefore=f"{phase['π0 base L12']:.3f}", MapPhasePiZeroAfter=f"{phase['π0 LIBERO L12']:.3f}",
         MapGoalPiZeroAfter=f"{goal['π0 LIBERO L12']:.0f}")  # fmt: skip
r2 = {n: u.suite_r2(n) for n in spaces}
before, after = [r2[b] for b, _a in pairs], [r2[a] for _b, a in pairs]
m.update(MapSuiteResnet=f"{100 * r2['ResNet-18 grid']:.0f}", MapSuiteBeforeMin=f"{100 * min(before):.0f}",
         MapSuiteBeforeMax=f"{100 * max(before):.0f}", MapSuiteAfterMin=f"{100 * min(after):.0f}",
         MapSuiteAfterMax=f"{100 * max(after):.0f}", MapSuiteDown=str(sum(r2[a] < r2[b] for b, a in pairs)))  # fmt: skip

## The kNN policy: hyper-parameter search and retry

In [ ]:
op = u.DATA["optuna"]
m.update(OptunaN=str(op["n_trials"]), OptunaAll=str(op["n_trials_all_studies"]),
         OptunaK=f"{100 * op['importance']['k']:.0f}", OptunaOne=f"{100 * op['importance']['one_per_episode']:.0f}")  # fmt: skip
retry = u.DATA["retry"]
others = [v for k, v in retry.items() if k != "base"]
m.update(RetryBase=u.f1(100 * retry["base"]), RetryMin=u.f1(100 * min(others)), RetryMax=u.f1(100 * max(others)))

In [ ]:
lines = ["% generated by poster/analysis/numbers.ipynb -- do not edit"]
lines += [f"\\newcommand{{\\{name}}}{{{value}}}" for name, value in m.items()]
(figs_path / "numbers.tex").write_text("\n".join(lines) + "\n")
print("\n".join(lines))